# Problem Statement:
# You are to enhance your 'deep research agent'—currently a relatively basic agent script or tool—by making it more agentic, autonomous, and capable of conducting substantive research.
# Clarifying Questions:
# When a user submits a deep research task or query, the agent should first generate and ask a few clarifying questions (e.g., 3 questions) about the query.
# These questions should be incorporated into subsequent searches and analysis.

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict, List
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

In [ ]:
# NEW: Pydantic models for clarifying questions
class ClarifyingQuestion(BaseModel):
    question: str = Field(description="A clarifying question to better understand the user's intent")
    purpose: str = Field(description="Why this question is important for the research")

class ClarifyingQuestions(BaseModel):
    questions: List[ClarifyingQuestion] = Field(description="List of clarifying questions to ask the user")

class QuestionResponse(BaseModel):
    question: str
    answer: str

In [ ]:
# NEW: Clarifying Questions Agent
CLARIFY_INSTRUCTIONS = """You are a research planning assistant. When given a research query, your job is to generate exactly 3 clarifying questions that will help you better understand what the user is looking for.

Consider:
- What specific aspects they want to focus on
- What level of detail they need
- What their intended use case is
- Any constraints or preferences they might have

Generate questions that will help narrow down the scope and improve the quality of the research."""

clarifying_agent = Agent(
    name="ClarifyingAgent",
    instructions=CLARIFY_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ClarifyingQuestions,
)


In [ ]:
# Original agents (keeping them as they were)
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
HOW_MANY_SEARCHES = 3

In [ ]:
# MODIFIED: Updated planner instructions to consider clarifying answers
PLANNER_INSTRUCTIONS = f"You are a helpful research assistant. Given a query and clarifying information from the user, \
come up with a set of web searches to perform to best answer the query. Take into account the user's specific \
requirements and preferences as indicated in their answers to clarifying questions. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)


In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com") # Change this to your verified email
    to_email = To("ed.donner@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [ ]:
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [ ]:
# NEW: Function to generate clarifying questions
async def generate_clarifying_questions(query: str) -> ClarifyingQuestions:
    """Generate clarifying questions for the user's query"""
    print("Generating clarifying questions...")
    result = await Runner.run(clarifying_agent, f"Research query: {query}")
    print(f"Generated {len(result.final_output.questions)} clarifying questions")
    return result.final_output

# NEW: Function to collect user answers (for Jupyter notebook interaction)
def collect_user_answers(clarifying_questions: ClarifyingQuestions) -> List[QuestionResponse]:
    """Collect user answers to clarifying questions"""
    print("\n" + "="*60)
    print("CLARIFYING QUESTIONS")
    print("="*60)
    print("To provide you with the best research, please answer these questions:")
    print()
    
    responses = []
    
    for i, q in enumerate(clarifying_questions.questions, 1):
        print(f"Question {i}: {q.question}")
        print(f"Purpose: {q.purpose}")
        print()
        
        # In a real implementation, you might want to use ipywidgets for better interaction
        # For now, we'll use a simple input approach
        answer = input("Your answer: ").strip()
        
        responses.append(QuestionResponse(question=q.question, answer=answer))
        print("-" * 40)
    
    print("Thank you! Now proceeding with the research...\n")
    return responses

In [ ]:
# MODIFIED: Enhanced plan_searches to consider clarifying answers
async def plan_searches(query: str, clarifying_responses: List[QuestionResponse] = None):
    """Use the planner_agent to plan which searches to run for the query"""
    print("Planning searches...")
    
    # Prepare input with clarifying information
    input_text = f"Query: {query}"
    
    if clarifying_responses:
        input_text += "\n\nUser's clarifying information:"
        for response in clarifying_responses:
            input_text += f"\nQ: {response.question}\nA: {response.answer}"
    
    result = await Runner.run(planner_agent, input_text)
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """Call search() for each item in the search plan"""
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """Use the search agent to run a web search for each item in the search plan"""
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

# MODIFIED: Enhanced write_report to consider clarifying answers
async def write_report(query: str, search_results: list[str], clarifying_responses: List[QuestionResponse] = None):
    """Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    
    input_text = f"Original query: {query}\nSummarized search results: {search_results}"
    
    if clarifying_responses:
        input_text += "\n\nUser's specific requirements:"
        for response in clarifying_responses:
            input_text += f"\nQ: {response.question}\nA: {response.answer}"
    
    result = await Runner.run(writer_agent, input_text)
    print("Finished writing report")
    return result.final_output

async def send_email_report(report: ReportData):
    """Use the email agent to send an email with the report"""
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report 

In [ ]:
# NEW: Enhanced main research function with clarifying questions
async def conduct_enhanced_research(query: str, ask_clarifying_questions: bool = True):
    """
    Conduct enhanced research with optional clarifying questions step
    
    Args:
        query: The research query
        ask_clarifying_questions: Whether to ask clarifying questions first
    """
    clarifying_responses = None
    
    if ask_clarifying_questions:
        # Step 1: Generate and ask clarifying questions
        clarifying_questions = await generate_clarifying_questions(query)
        clarifying_responses = collect_user_answers(clarifying_questions)
    
    # Step 2: Plan searches (considering clarifying answers)
    search_plan = await plan_searches(query, clarifying_responses)
    
    # Step 3: Perform searches
    search_results = await perform_searches(search_plan)
    
    # Step 4: Write report (considering clarifying answers)
    report = await write_report(query, search_results, clarifying_responses)
    
    # Step 5: Send email
    await send_email_report(report)
    
    return report

In [ ]:
# Example usage with clarifying questions
query = "Latest AI Agent frameworks in 2025"

with trace("Enhanced Research with Clarifying Questions"):
    print("Starting enhanced research with clarifying questions...")
    report = await conduct_enhanced_research(query, ask_clarifying_questions=True)
    print("Hooray! Enhanced research completed!")
    
    # Display the report
    display(Markdown("## Research Report"))
    display(Markdown(report.markdown_report))

In [ ]:
# Alternative: Run without clarifying questions (original behavior)
# with trace("Enhanced Research without Clarifying Questions"):
#     print("Starting research without clarifying questions...")
#     report = await conduct_enhanced_research(query, ask_clarifying_questions=False)
#     print("Hooray! Research completed!")

In [ ]:
# Example of manual clarifying questions for testing
async def test_with_manual_answers():
    """Test the system with predefined answers"""
    query = "Latest AI Agent frameworks in 2025"
    
    # Generate questions
    clarifying_questions = await generate_clarifying_questions(query)
    
    # Provide manual answers for testing
    manual_responses = [
        QuestionResponse(
            question=clarifying_questions.questions[0].question,
            answer="I'm particularly interested in open-source frameworks that are actively maintained and have good documentation."
        ),
        QuestionResponse(
            question=clarifying_questions.questions[1].question,
            answer="I need this for building production-ready applications, so I care about performance, scalability, and enterprise features."
        ),
        QuestionResponse(
            question=clarifying_questions.questions[2].question,
            answer="I'm most interested in frameworks that work well with Python and have good integration with popular LLMs like OpenAI and Anthropic."
        )
    ]
    
    with trace("Test with Manual Answers"):
        print("Testing with manual answers...")
        search_plan = await plan_searches(query, manual_responses)
        search_results = await perform_searches(search_plan)
        report = await write_report(query, search_results, manual_responses)
        
        display(Markdown("## Test Report with Manual Answers"))
        display(Markdown(report.markdown_report[:500] + "..."))
        
        return report 

In [ ]:
# Uncomment to run the test
await test_with_manual_answers()

# How to use 
- With Clarifying Questions (Interactive):
- `report = await conduct_enhanced_research(query, ask_clarifying_questions=True)`

- Without Clarifying Questions (Original behavior):
- `report = await conduct_enhanced_research(query, ask_clarifying_questions=False)`

- Testing Mode
- `await test_with_manual_answers()`
